# KuaiRand Data Understanding and EDA

This notebook uses PySpark/Spark SQL for scalable exploration. The intended flow is:

1. Inspect raw CSV files and schemas.
2. Convert CSV to Parquet in the bronze layer.
3. Use Parquet for row counts, missing values, duplicates, timestamps, interaction flags, and basic user/item/temporal statistics.
4. Prepare reusable columns for later session analysis and preference-drift analysis.

In [67]:
from pathlib import Path
import importlib
import os
import sys

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 4g pyspark-shell")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import SparkSession, functions as F
from pyspark.sql import Window

import recommender.data.kuairand as kuairand_module
import recommender.spark as spark_module

kuairand_module = importlib.reload(kuairand_module)
spark_module = importlib.reload(spark_module)
read_csv = kuairand_module.read_csv
get_spark = spark_module.get_spark

try:
    spark = get_spark("kuairand-eda", reset=True)
except TypeError:
    # Fallback for an already-running kernel that still has an older helper loaded.
    spark = (
        SparkSession.builder.master("local[*]")
        .appName("kuairand-eda")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.local.dir", str(PROJECT_ROOT / "data/spark-tmp"))
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.pyspark.python", sys.executable)
        .config("spark.pyspark.driver.python", sys.executable)
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("ERROR")
print(f"Python executable: {sys.executable}")
print(f"Spark {spark.version} | master={spark.sparkContext.master} | driver={spark.sparkContext.getConf().get('spark.driver.host')}")
spark


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 23:27:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/22 23:27:23 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
26/09/22 23:27:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/22 23:27:23 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/22 23:27:23 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Python executable: /Users/khoatran/miniconda3/envs/recsys/bin/python
Spark 3.5.6 | master=local[*] | driver=127.0.0.1


## Configure Paths

The notebook auto-detects raw CSV files in `data/raw/kuairand`, then `data/raw`, then `data`. Parquet outputs go to `data/bronze/kuairand`.

If you changed Spark memory or local IP settings after the first cell already ran, restart the kernel before continuing. Spark only applies those settings when the JVM starts.

In [68]:
RAW_CSV_CANDIDATES = [
    PROJECT_ROOT / "data/raw/kuairand",
    PROJECT_ROOT / "data/raw",
    PROJECT_ROOT / "data",
]
RAW_CSV_DIR = next((p for p in RAW_CSV_CANDIDATES if list(p.glob("*.csv"))), RAW_CSV_CANDIDATES[0])
BRONZE_PARQUET_DIR = PROJECT_ROOT / "data/bronze/kuairand"

print(f"Raw CSV directory: {RAW_CSV_DIR}")
print(f"Bronze Parquet directory: {BRONZE_PARQUET_DIR}")

USER_COL = "user_id"
ITEM_COL = "video_id"
DATE_COL = "date"
HOURMIN_COL = "hourmin"
TIME_MS_COL = "time_ms"

INTERACTION_COLS = [
    "is_click",
    "is_like",
    "is_follow",
    "is_comment",
    "is_forward",
    "is_hate",
    "long_view",
    "is_profile_enter",
]


Raw CSV directory: /Users/khoatran/coding/recsys/data/raw
Bronze Parquet directory: /Users/khoatran/coding/recsys/data/bronze/kuairand


## Inspect Raw Files

In [69]:
csv_files = sorted(RAW_CSV_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {RAW_CSV_DIR}. Put KuaiRand CSVs under data/raw or data/raw/kuairand.")
[(p.name, round(p.stat().st_size / 1024 / 1024, 3)) for p in csv_files]


[('log_random_4_22_to_5_08_1k.csv', 2.943),
 ('log_standard_4_08_to_4_21_1k.csv', 356.36),
 ('log_standard_4_22_to_5_08_1k.csv', 469.497),
 ('user_features_1k.csv', 0.122),
 ('video_features_basic_1k.csv', 359.428),
 ('video_features_statistic_1k.csv', 3214.482)]

In [70]:
raw_tables = {}
for path in csv_files:
    name = path.stem
    df = read_csv(spark, path)
    raw_tables[name] = df
    print(f"\n{name}")
    df.printSchema()
    df.show(5, truncate=False)



log_random_4_22_to_5_08_1k
root
 |-- user_id: integer (nullable = true)
 |-- video_id: integer (nullable = true)
 |-- date: integer (nullable = true)
 |-- hourmin: integer (nullable = true)
 |-- time_ms: long (nullable = true)
 |-- is_click: integer (nullable = true)
 |-- is_like: integer (nullable = true)
 |-- is_follow: integer (nullable = true)
 |-- is_comment: integer (nullable = true)
 |-- is_forward: integer (nullable = true)
 |-- is_hate: integer (nullable = true)
 |-- long_view: integer (nullable = true)
 |-- play_time_ms: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- profile_stay_time: integer (nullable = true)
 |-- comment_stay_time: integer (nullable = true)
 |-- is_profile_enter: integer (nullable = true)
 |-- is_rand: integer (nullable = true)
 |-- tab: integer (nullable = true)

+-------+--------+--------+-------+-------------+--------+-------+---------+----------+----------+-------+---------+------------+-----------+-----------------+-------

## Convert CSV to Parquet

Parquet is the default format for analysis because it is columnar, typed, compressed, and much faster for repeated Spark scans than CSV.

In [71]:
BRONZE_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

for path in csv_files:
    name = path.stem
    target = BRONZE_PARQUET_DIR / name
    print(f"converting {path.name} -> {target}")
    df = read_csv(spark, path)
    df.write.mode("overwrite").parquet(str(target))
    print(f"wrote {target}")

spark.catalog.clearCache()


converting log_random_4_22_to_5_08_1k.csv -> /Users/khoatran/coding/recsys/data/bronze/kuairand/log_random_4_22_to_5_08_1k
wrote /Users/khoatran/coding/recsys/data/bronze/kuairand/log_random_4_22_to_5_08_1k
converting log_standard_4_08_to_4_21_1k.csv -> /Users/khoatran/coding/recsys/data/bronze/kuairand/log_standard_4_08_to_4_21_1k


wrote /Users/khoatran/coding/recsys/data/bronze/kuairand/log_standard_4_08_to_4_21_1k
converting log_standard_4_22_to_5_08_1k.csv -> /Users/khoatran/coding/recsys/data/bronze/kuairand/log_standard_4_22_to_5_08_1k


wrote /Users/khoatran/coding/recsys/data/bronze/kuairand/log_standard_4_22_to_5_08_1k
converting user_features_1k.csv -> /Users/khoatran/coding/recsys/data/bronze/kuairand/user_features_1k
wrote /Users/khoatran/coding/recsys/data/bronze/kuairand/user_features_1k
converting video_features_basic_1k.csv -> /Users/khoatran/coding/recsys/data/bronze/kuairand/video_features_basic_1k


wrote /Users/khoatran/coding/recsys/data/bronze/kuairand/video_features_basic_1k
converting video_features_statistic_1k.csv -> /Users/khoatran/coding/recsys/data/bronze/kuairand/video_features_statistic_1k


wrote /Users/khoatran/coding/recsys/data/bronze/kuairand/video_features_statistic_1k


In [72]:
parquet_tables = {}
for path in sorted(p for p in BRONZE_PARQUET_DIR.iterdir() if p.is_dir()):
    parquet_tables[path.name] = spark.read.parquet(str(path))

list(parquet_tables.keys())

['log_random_4_22_to_5_08_1k',
 'log_standard_4_08_to_4_21_1k',
 'log_standard_4_22_to_5_08_1k',
 'user_features_1k',
 'video_features_basic_1k',
 'video_features_statistic_1k']

## Understand Each KuaiRand Table

Before session or preference analysis, inspect each data type on its own: interaction logs, user features, video metadata, and video historical statistics.

In [73]:
table_groups = {
    "interaction_logs": sorted([name for name in parquet_tables if name.startswith("log_")]),
    "user_features": sorted([name for name in parquet_tables if name.startswith("user_features")]),
    "video_basic": sorted([name for name in parquet_tables if name.startswith("video_features_basic")]),
    "video_statistics": sorted([name for name in parquet_tables if name.startswith("video_features_statistic")]),
}

for group_name, names in table_groups.items():
    print(f"{group_name}: {names}")


interaction_logs: ['log_random_4_22_to_5_08_1k', 'log_standard_4_08_to_4_21_1k', 'log_standard_4_22_to_5_08_1k']
user_features: ['user_features_1k']
video_basic: ['video_features_basic_1k']
video_statistics: ['video_features_statistic_1k']


In [74]:
table_profile_rows = []
for group_name, names in table_groups.items():
    for name in names:
        df = parquet_tables[name]
        table_profile_rows.append((group_name, name, df.count(), len(df.columns), ", ".join(df.columns[:8])))

spark.createDataFrame(
    table_profile_rows,
    ["group", "table", "rows", "columns", "first_columns"],
).orderBy("group", "table").show(truncate=False)


+----------------+----------------------------+-------+-------+------------------------------------------------------------------------------------------------------------------------------------------+
|group           |table                       |rows   |columns|first_columns                                                                                                                             |
+----------------+----------------------------+-------+-------+------------------------------------------------------------------------------------------------------------------------------------------+
|interaction_logs|log_random_4_22_to_5_08_1k  |43028  |19     |user_id, video_id, date, hourmin, time_ms, is_click, is_like, is_follow                                                                   |
|interaction_logs|log_standard_4_08_to_4_21_1k|5055984|19     |user_id, video_id, date, hourmin, time_ms, is_click, is_like, is_follow                                                      

In [75]:
def inspect_table(name, n=5):
    df = parquet_tables[name]
    print(f"\n{name}")
    df.printSchema()
    df.show(n, truncate=False)

for names in table_groups.values():
    for name in names:
        inspect_table(name, n=3)



log_random_4_22_to_5_08_1k
root
 |-- user_id: integer (nullable = true)
 |-- video_id: integer (nullable = true)
 |-- date: integer (nullable = true)
 |-- hourmin: integer (nullable = true)
 |-- time_ms: long (nullable = true)
 |-- is_click: integer (nullable = true)
 |-- is_like: integer (nullable = true)
 |-- is_follow: integer (nullable = true)
 |-- is_comment: integer (nullable = true)
 |-- is_forward: integer (nullable = true)
 |-- is_hate: integer (nullable = true)
 |-- long_view: integer (nullable = true)
 |-- play_time_ms: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- profile_stay_time: integer (nullable = true)
 |-- comment_stay_time: integer (nullable = true)
 |-- is_profile_enter: integer (nullable = true)
 |-- is_rand: integer (nullable = true)
 |-- tab: integer (nullable = true)

+-------+--------+--------+-------+-------------+--------+-------+---------+----------+----------+-------+---------+------------+-----------+-----------------+-------

### Missing Values by Table

In [76]:
def missing_profile(df, table_name):
    total = df.count()
    exprs = [F.sum(F.col(c).isNull().cast("long")).alias(c) for c in df.columns]
    row = df.agg(*exprs).collect()[0].asDict()
    rows = [(table_name, col, missing, missing / total if total else None) for col, missing in row.items() if missing]
    if not rows:
        rows = [(table_name, "<none>", 0, 0.0)]
    return rows

missing_rows = []
for names in table_groups.values():
    for name in names:
        missing_rows.extend(missing_profile(parquet_tables[name], name))

spark.createDataFrame(missing_rows, ["table", "column", "missing", "missing_rate"]).orderBy(
    "table", F.desc("missing")
).show(100, truncate=False)


+----------------------------+--------------+-------+---------------------+
|table                       |column        |missing|missing_rate         |
+----------------------------+--------------+-------+---------------------+
|log_random_4_22_to_5_08_1k  |<none>        |0      |0.0                  |
|log_standard_4_08_to_4_21_1k|<none>        |0      |0.0                  |
|log_standard_4_22_to_5_08_1k|<none>        |0      |0.0                  |
|user_features_1k            |onehot_feat14 |33     |0.033                |
|user_features_1k            |onehot_feat17 |33     |0.033                |
|user_features_1k            |onehot_feat15 |33     |0.033                |
|user_features_1k            |onehot_feat16 |33     |0.033                |
|user_features_1k            |onehot_feat12 |33     |0.033                |
|user_features_1k            |onehot_feat4  |33     |0.033                |
|user_features_1k            |onehot_feat13 |33     |0.033                |
|video_featu

### User Feature Tables

In [77]:
for name in table_groups["user_features"]:
    df = parquet_tables[name]
    print(f"\n{name}: users={df.select(USER_COL).distinct().count()}")
    df.select(
        "follow_user_num", "fans_user_num", "friend_user_num", "register_days"
    ).summary("count", "min", "25%", "50%", "75%", "max").show(truncate=False)
    for col in [
        "user_active_degree", "is_lowactive_period", "is_live_streamer", "is_video_author",
        "follow_user_num_range", "fans_user_num_range", "friend_user_num_range", "register_days_range",
    ]:
        print(f"\nTop values for {col}")
        df.groupBy(col).count().orderBy(F.desc("count")).show(20, truncate=False)



user_features_1k: users=1000
+-------+---------------+-------------+---------------+-------------+
|summary|follow_user_num|fans_user_num|friend_user_num|register_days|
+-------+---------------+-------------+---------------+-------------+
|count  |1000           |1000         |1000           |1000         |
|min    |0              |0            |0              |22           |
|25%    |46             |9            |2              |698          |
|50%    |160            |43           |14             |1233         |
|75%    |419            |198          |77             |1751         |
|max    |5000           |156158       |4690           |3187         |
+-------+---------------+-------------+---------------+-------------+


Top values for user_active_degree
+------------------+-----+
|user_active_degree|count|
+------------------+-----+
|full_active       |640  |
|high_active       |233  |
|middle_active     |83   |
|2_14_day_new      |23   |
|low_active        |17   |
|30day_retention  

### Video Basic Feature Tables

In [78]:
for name in table_groups["video_basic"]:
    df = parquet_tables[name]
    video_count = df.select(ITEM_COL).distinct().count()
    author_count = df.select("author_id").distinct().count()
    print(f"\n{name}: videos={video_count}, authors={author_count}")
    df.select("video_duration", "server_width", "server_height").summary(
        "count", "min", "25%", "50%", "75%", "max"
    ).show(truncate=False)
    for col in ["video_type", "upload_type", "visible_status", "music_type", "tag"]:
        print(f"\nTop values for {col}")
        df.groupBy(col).count().orderBy(F.desc("count")).show(20, truncate=False)



video_features_basic_1k: videos=4371868, authors=1407475


+-------+--------------+------------+-------------+
|summary|video_duration|server_width|server_height|
+-------+--------------+------------+-------------+
|count  |3798811       |4371861     |4371861      |
|min    |40.0          |48.0        |33.0         |
|25%    |12066.0       |720.0       |1280.0       |
|50%    |35100.0       |720.0       |1280.0       |
|75%    |97533.0       |720.0       |1280.0       |
|max    |1.651308E7    |8394.0      |11826.0      |
+-------+--------------+------------+-------------+


Top values for video_type
+----------+-------+
|video_type|count  |
+----------+-------+
|NORMAL    |4361638|
|AD        |9985   |
|UNKNOWN   |245    |
+----------+-------+


Top values for upload_type
+----------------------+-------+
|upload_type           |count  |
+----------------------+-------+
|LongImport            |1108258|
|ShortImport           |872196 |
|Kmovie                |777186 |
|Web                   |428111 |
|LongPicture           |323304 |
|LongCamera 

### Video Historical Statistic Tables

In [79]:
video_stat_focus_cols = [
    "counts", "show_cnt", "show_user_num", "play_cnt", "play_user_num", "play_duration",
    "valid_play_cnt", "long_time_play_cnt", "short_time_play_cnt", "play_progress",
    "like_cnt", "comment_cnt", "follow_cnt", "share_cnt", "collect_cnt", "report_cnt",
]

for name in table_groups["video_statistics"]:
    df = parquet_tables[name]
    existing_cols = [c for c in video_stat_focus_cols if c in df.columns]
    print(f"\n{name}: videos={df.select(ITEM_COL).distinct().count()}")
    df.select(*existing_cols).summary("count", "min", "25%", "50%", "75%", "max").show(truncate=False)
    print("\nTop videos by play_cnt")
    df.select(ITEM_COL, "play_cnt", "play_user_num", "play_duration", "like_cnt", "comment_cnt", "share_cnt").orderBy(
        F.desc("play_cnt")
    ).show(20, truncate=False)



video_features_statistic_1k: videos=4371868


+-------+-------+------------------+------------------+------------------+------------------+--------------------+-----------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+-----------------+----------+
|summary|counts |show_cnt          |show_user_num     |play_cnt          |play_user_num     |play_duration       |valid_play_cnt   |long_time_play_cnt|short_time_play_cnt|play_progress     |like_cnt          |comment_cnt       |follow_cnt        |share_cnt         |collect_cnt      |report_cnt|
+-------+-------+------------------+------------------+------------------+------------------+--------------------+-----------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+-----------------+----------+
|count  |4371868|4371868           |4371868           |4371868           |4371868           |4371868            

### Interaction Log Tables

In [80]:
for name in table_groups["interaction_logs"]:
    df = parquet_tables[name]
    print(f"\n{name}")
    df.agg(
        F.count("*").alias("rows"),
        F.countDistinct(USER_COL).alias("users"),
        F.countDistinct(ITEM_COL).alias("videos"),
        F.min(DATE_COL).alias("min_date"),
        F.max(DATE_COL).alias("max_date"),
        *[F.sum(F.col(c).cast("long")).alias(c) for c in INTERACTION_COLS if c in df.columns],
    ).show(truncate=False)
    for col in ["is_rand", "tab"]:
        if col in df.columns:
            print(f"\nDistribution for {col}")
            df.groupBy(col).count().orderBy(col).show(truncate=False)



log_random_4_22_to_5_08_1k
+-----+-----+------+--------+--------+--------+-------+---------+----------+----------+-------+---------+----------------+
|rows |users|videos|min_date|max_date|is_click|is_like|is_follow|is_comment|is_forward|is_hate|long_view|is_profile_enter|
+-----+-----+------+--------+--------+--------+-------+---------+----------+----------+-------+---------+----------------+
|43028|1000 |7388  |20220422|20220508|7492    |239    |8        |13        |28        |35     |3621     |198             |
+-----+-----+------+--------+--------+--------+-------+---------+----------+----------+-------+---------+----------------+


Distribution for is_rand
+-------+-----+
|is_rand|count|
+-------+-----+
|1      |43028|
+-------+-----+


Distribution for tab
+---+-----+
|tab|count|
+---+-----+
|1  |42636|
|2  |73   |
|11 |304  |
|14 |15   |
+---+-----+


log_standard_4_08_to_4_21_1k


+-------+-----+-------+--------+--------+--------+-------+---------+----------+----------+-------+---------+----------------+
|rows   |users|videos |min_date|max_date|is_click|is_like|is_follow|is_comment|is_forward|is_hate|long_view|is_profile_enter|
+-------+-----+-------+--------+--------+--------+-------+---------+----------+----------+-------+---------+----------------+
|5055984|983  |2119510|20220408|20220421|1917934 |76108  |5722     |12391     |3963      |8727   |1332063  |90257           |
+-------+-----+-------+--------+--------+--------+-------+---------+----------+----------+-------+---------+----------------+


Distribution for is_rand
+-------+-------+
|is_rand|count  |
+-------+-------+
|0      |5055984|
+-------+-------+


Distribution for tab
+---+-------+
|tab|count  |
+---+-------+
|0  |1066194|
|1  |3286302|
|2  |183556 |
|3  |19720  |
|4  |402534 |
|5  |6991   |
|6  |66045  |
|7  |1696   |
|8  |7712   |
|9  |1132   |
|10 |588    |
|11 |10344  |
|12 |3164   |
|14 |6

+-------+-----+-------+--------+--------+--------+-------+---------+----------+----------+-------+---------+----------------+
|rows   |users|videos |min_date|max_date|is_click|is_like|is_follow|is_comment|is_forward|is_hate|long_view|is_profile_enter|
+-------+-----+-------+--------+--------+--------+-------+---------+----------+----------+-------+---------+----------------+
|6657061|1000 |2664050|20220422|20220508|2511906 |106734 |5676     |18735     |5228      |2129   |1737398  |119432          |
+-------+-----+-------+--------+--------+--------+-------+---------+----------+----------+-------+---------+----------------+


Distribution for is_rand
+-------+-------+
|is_rand|count  |
+-------+-------+
|0      |6657061|
+-------+-------+


Distribution for tab
+---+-------+
|tab|count  |
+---+-------+
|0  |1341158|
|1  |4431299|
|2  |218737 |
|3  |17698  |
|4  |492851 |
|5  |9433   |
|6  |117358 |
|7  |2137   |
|8  |9718   |
|9  |779    |
|10 |364    |
|11 |7230   |
|12 |7843   |
|13 |1

## Pick Interaction Tables

KuaiRand has log tables plus user/video feature tables. This cell unions all log-like tables with matching columns for interaction-level analysis.

In [81]:
log_names = [name for name, df in parquet_tables.items() if {USER_COL, ITEM_COL}.issubset(df.columns) and name.startswith("log_")]
log_names

['log_random_4_22_to_5_08_1k',
 'log_standard_4_08_to_4_21_1k',
 'log_standard_4_22_to_5_08_1k']

In [82]:
logs = None
for name in log_names:
    df = parquet_tables[name].withColumn("source_table", F.lit(name))
    logs = df if logs is None else logs.unionByName(df, allowMissingColumns=True)

logs.printSchema()
logs.show(5, truncate=False)

root
 |-- user_id: integer (nullable = true)
 |-- video_id: integer (nullable = true)
 |-- date: integer (nullable = true)
 |-- hourmin: integer (nullable = true)
 |-- time_ms: long (nullable = true)
 |-- is_click: integer (nullable = true)
 |-- is_like: integer (nullable = true)
 |-- is_follow: integer (nullable = true)
 |-- is_comment: integer (nullable = true)
 |-- is_forward: integer (nullable = true)
 |-- is_hate: integer (nullable = true)
 |-- long_view: integer (nullable = true)
 |-- play_time_ms: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- profile_stay_time: integer (nullable = true)
 |-- comment_stay_time: integer (nullable = true)
 |-- is_profile_enter: integer (nullable = true)
 |-- is_rand: integer (nullable = true)
 |-- tab: integer (nullable = true)
 |-- source_table: string (nullable = false)

+-------+--------+--------+-------+-------------+--------+-------+---------+----------+----------+-------+---------+------------+-----------+--------

## Row Counts, Users, Items

In [83]:
table_counts = []
for name, df in parquet_tables.items():
    table_counts.append((name, df.count(), len(df.columns)))

spark.createDataFrame(table_counts, ["table", "rows", "columns"]).orderBy("table").show(truncate=False)

+----------------------------+-------+-------+
|table                       |rows   |columns|
+----------------------------+-------+-------+
|log_random_4_22_to_5_08_1k  |43028  |19     |
|log_standard_4_08_to_4_21_1k|5055984|19     |
|log_standard_4_22_to_5_08_1k|6657061|19     |
|user_features_1k            |1000   |31     |
|video_features_basic_1k     |4371868|12     |
|video_features_statistic_1k |4371868|52     |
+----------------------------+-------+-------+



In [84]:
logs.agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("users"),
    F.countDistinct(ITEM_COL).alias("items"),
    F.countDistinct("source_table").alias("log_tables"),
).show(truncate=False)

+------------+-----+-------+----------+
|interactions|users|items  |log_tables|
+------------+-----+-------+----------+
|11756073    |1000 |4371868|3         |
+------------+-----+-------+----------+



## Missing Values and Duplicates

In [85]:
def missing_summary(df):
    exprs = [F.sum(F.col(c).isNull().cast("long")).alias(c) for c in df.columns]
    return df.agg(*exprs)

missing_summary(logs).show(vertical=True, truncate=False)

-RECORD 0----------------
 user_id           | 0   
 video_id          | 0   
 date              | 0   
 hourmin           | 0   
 time_ms           | 0   
 is_click          | 0   
 is_like           | 0   
 is_follow         | 0   
 is_comment        | 0   
 is_forward        | 0   
 is_hate           | 0   
 long_view         | 0   
 play_time_ms      | 0   
 duration_ms       | 0   
 profile_stay_time | 0   
 comment_stay_time | 0   
 is_profile_enter  | 0   
 is_rand           | 0   
 tab               | 0   
 source_table      | 0   



In [86]:
duplicate_keys = [USER_COL, ITEM_COL, DATE_COL, HOURMIN_COL, TIME_MS_COL]
available_duplicate_keys = [c for c in duplicate_keys if c in logs.columns]

# Exact duplicate checks across the full interaction log are shuffle-heavy.
# For EDA, inspect duplicate candidates on a deterministic sample first.
DUPLICATE_SAMPLE_FRACTION = 0.05
logs_sample = logs.sample(withReplacement=False, fraction=DUPLICATE_SAMPLE_FRACTION, seed=42)

logs_sample.groupBy(*available_duplicate_keys).count().where(F.col("count") > 1).orderBy(
    F.desc("count")
).show(20, truncate=False)


+-------+--------+--------+-------+-------------+-----+
|user_id|video_id|date    |hourmin|time_ms      |count|
+-------+--------+--------+-------+-------------+-----+
|406    |1792598 |20220414|600    |1649889391171|2    |
|444    |3897869 |20220416|1200   |1650081991205|2    |
|410    |2955737 |20220415|1500   |1650006517317|2    |
|278    |1374344 |20220412|1000   |1649730788304|2    |
|448    |3278752 |20220416|1800   |1650105594635|2    |
|143    |4195784 |20220410|100    |1649525780437|2    |
|424    |1481884 |20220409|1600   |1649491665872|2    |
|175    |3333212 |20220419|0      |1650297781083|2    |
|441    |2472331 |20220420|2200   |1650463360728|2    |
|212    |4152600 |20220415|1300   |1650000124124|2    |
|427    |3285460 |20220418|2000   |1650284249146|2    |
|221    |3618812 |20220421|2100   |1650545725911|2    |
|372    |870175  |20220409|700    |1649458821906|2    |
|694    |343411  |20220416|1200   |1650081757843|2    |
|448    |2248152 |20220419|2000   |1650371730662

## Timestamp and Interaction Types

In [87]:
def with_event_time(df):
    out = df
    if DATE_COL in out.columns:
        out = out.withColumn("event_date", F.to_date(F.col(DATE_COL).cast("string"), "yyyyMMdd"))
    if TIME_MS_COL in out.columns:
        out = out.withColumn("event_ts", F.to_timestamp(F.from_unixtime((F.col(TIME_MS_COL) / 1000).cast("long"))))
    if HOURMIN_COL in out.columns:
        out = out.withColumn("hourmin_str", F.lpad(F.col(HOURMIN_COL).cast("string"), 4, "0"))
        out = out.withColumn("event_hour", F.substring("hourmin_str", 1, 2).cast("int"))
        out = out.withColumn("event_minute", F.substring("hourmin_str", 3, 2).cast("int"))
    return out

logs_ts = with_event_time(logs)
logs_ts.select(DATE_COL, HOURMIN_COL, TIME_MS_COL, "event_date", "event_ts", "event_hour", "event_minute").show(10, truncate=False)

+--------+-------+-------------+----------+-------------------+----------+------------+
|date    |hourmin|time_ms      |event_date|event_ts           |event_hour|event_minute|
+--------+-------+-------------+----------+-------------------+----------+------------+
|20220430|1800   |1651314030792|2022-04-30|2022-04-30 10:20:30|18        |0           |
|20220502|1200   |1651466607423|2022-05-02|2022-05-02 04:43:27|12        |0           |
|20220502|1700   |1651481542743|2022-05-02|2022-05-02 08:52:22|17        |0           |
|20220502|1800   |1651488577163|2022-05-02|2022-05-02 10:49:37|18        |0           |
|20220503|800    |1651535940163|2022-05-03|2022-05-02 23:59:00|8         |0           |
|20220504|900    |1651625898330|2022-05-04|2022-05-04 00:58:18|9         |0           |
|20220504|900    |1651628052625|2022-05-04|2022-05-04 01:34:12|9         |0           |
|20220506|700    |1651793743336|2022-05-06|2022-05-05 23:35:43|7         |0           |
|20220506|700    |1651793764164|

In [88]:
logs_ts.agg(
    F.min("event_date").alias("min_date"),
    F.max("event_date").alias("max_date"),
    F.min("event_ts").alias("min_ts"),
    F.max("event_ts").alias("max_ts"),
).show(truncate=False)

+----------+----------+-------------------+-------------------+
|min_date  |max_date  |min_ts             |max_ts             |
+----------+----------+-------------------+-------------------+
|2022-04-08|2022-05-08|2022-04-07 14:00:03|2022-05-08 15:52:09|
+----------+----------+-------------------+-------------------+



In [89]:
existing_interaction_cols = [c for c in INTERACTION_COLS if c in logs_ts.columns]
interaction_summary = logs_ts.agg(*[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols])
interaction_summary.show(truncate=False)

+--------+-------+---------+----------+----------+-------+---------+----------------+
|is_click|is_like|is_follow|is_comment|is_forward|is_hate|long_view|is_profile_enter|
+--------+-------+---------+----------+----------+-------+---------+----------------+
|4437332 |183081 |11406    |31139     |9219      |10891  |3073082  |209887          |
+--------+-------+---------+----------+----------+-------+---------+----------------+



## Basic User, Item, and Temporal Statistics

In [90]:
user_stats = logs_ts.groupBy(USER_COL).agg(
    F.count("*").alias("interactions"),
    F.countDistinct(ITEM_COL).alias("distinct_items"),
    *[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols],
)

user_stats.orderBy(F.desc("interactions")).show(20, truncate=False)

+-------+------------+--------------+--------+-------+---------+----------+----------+-------+---------+----------------+
|user_id|interactions|distinct_items|is_click|is_like|is_follow|is_comment|is_forward|is_hate|long_view|is_profile_enter|
+-------+------------+--------------+--------+-------+---------+----------+----------+-------+---------+----------------+
|879    |127647      |124459        |7903    |50     |6        |52        |13        |0      |6037     |374             |
|793    |102101      |91744         |10008   |153    |5        |129       |45        |1      |4746     |2440            |
|909    |82930       |80424         |9514    |122    |3        |30        |23        |0      |6748     |174             |
|725    |76986       |73426         |8033    |82     |8        |31        |39        |0      |2877     |87              |
|71     |72471       |63558         |9769    |86     |16       |70        |29        |0      |5207     |1661            |
|75     |68119       |65

In [91]:
item_stats = logs_ts.groupBy(ITEM_COL).agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("distinct_users"),
    *[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols],
)

item_stats.orderBy(F.desc("interactions")).show(20, truncate=False)

+--------+------------+--------------+--------+-------+---------+----------+----------+-------+---------+----------------+
|video_id|interactions|distinct_users|is_click|is_like|is_follow|is_comment|is_forward|is_hate|long_view|is_profile_enter|
+--------+------------+--------------+--------+-------+---------+----------+----------+-------+---------+----------------+
|26075   |585         |385           |23      |5      |2        |0         |2         |0      |10       |0               |
|3892658 |431         |390           |262     |3      |0        |2         |0         |0      |108      |19              |
|160718  |414         |378           |265     |2      |0        |2         |0         |0      |245      |2               |
|111390  |408         |398           |278     |3      |0        |0         |0         |0      |243      |3               |
|4066465 |390         |361           |257     |2      |0        |1         |0         |0      |221      |1               |
|371172  |387   

In [92]:
logs_ts.groupBy("event_date").agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("users"),
    F.countDistinct(ITEM_COL).alias("items"),
).orderBy("event_date").show(60, truncate=False)

+----------+------------+-----+------+
|event_date|interactions|users|items |
+----------+------------+-----+------+
|2022-04-08|358009      |885  |219519|
|2022-04-09|400301      |901  |245162|
|2022-04-10|404561      |900  |246792|
|2022-04-11|370216      |870  |230550|
|2022-04-12|345469      |849  |217151|
|2022-04-13|346408      |848  |220199|
|2022-04-14|326469      |860  |207471|
|2022-04-15|361879      |899  |226594|
|2022-04-16|389039      |922  |238780|
|2022-04-17|396844      |908  |249210|
|2022-04-18|333779      |870  |215459|
|2022-04-19|337975      |868  |211887|
|2022-04-20|339479      |869  |211278|
|2022-04-21|345556      |868  |216486|
|2022-04-22|369464      |919  |226503|
|2022-04-23|422493      |936  |255629|
|2022-04-24|350623      |891  |217746|
|2022-04-25|343012      |878  |212512|
|2022-04-26|352000      |882  |214929|
|2022-04-27|350750      |883  |212965|
|2022-04-28|346852      |886  |214801|
|2022-04-29|395321      |941  |241564|
|2022-04-30|440549      |

In [93]:
logs_ts.groupBy("event_hour").agg(
    F.count("*").alias("interactions"),
    F.countDistinct(USER_COL).alias("users"),
).orderBy("event_hour").show(24, truncate=False)

+----------+------------+-----+
|event_hour|interactions|users|
+----------+------------+-----+
|0         |369849      |799  |
|1         |234784      |662  |
|2         |141585      |548  |
|3         |103967      |486  |
|4         |104045      |474  |
|5         |152887      |545  |
|6         |272429      |759  |
|7         |366291      |878  |
|8         |434070      |929  |
|9         |475243      |949  |
|10        |518498      |972  |
|11        |554671      |970  |
|12        |623105      |983  |
|13        |620542      |983  |
|14        |562484      |974  |
|15        |586800      |969  |
|16        |612563      |967  |
|17        |642702      |978  |
|18        |698474      |980  |
|19        |729133      |988  |
|20        |781023      |979  |
|21        |797194      |989  |
|22        |784715      |968  |
|23        |589019      |921  |
+----------+------------+-----+



## Session and Preference-Drift Preparation

These columns are not final features yet. They prepare the interaction log for later sessionization and short-term preference-drift analysis.

In [94]:
SESSION_GAP_MINUTES = 30

# Session analysis should focus on meaningful watch/click events, not every impression row.
# This keeps the session view interpretable and avoids huge local window shuffles.
session_base = logs_ts.where(
    (F.col("is_click") == 1)
    | (F.col("long_view") == 1)
    | (F.col("play_time_ms") > 0)
)

w_user_time = Window.partitionBy(USER_COL).orderBy("event_ts")

logs_sequence = (
    session_base
    .withColumn("prev_event_ts", F.lag("event_ts").over(w_user_time))
    .withColumn("gap_seconds", F.col("event_ts").cast("long") - F.col("prev_event_ts").cast("long"))
    .withColumn(
        "is_new_session",
        F.when(F.col("prev_event_ts").isNull(), F.lit(1))
        .when(F.col("gap_seconds") > SESSION_GAP_MINUTES * 60, F.lit(1))
        .otherwise(F.lit(0)),
    )
    .withColumn("session_index", F.sum("is_new_session").over(w_user_time.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn("session_id", F.concat_ws("_", F.col(USER_COL).cast("string"), F.col("session_index").cast("string")))
)

session_base.agg(F.count("*").alias("session_candidate_events")).show(truncate=False)
logs_sequence.select(USER_COL, ITEM_COL, "event_ts", "gap_seconds", "session_id", *existing_interaction_cols).show(20, truncate=False)


+------------------------+
|session_candidate_events|
+------------------------+
|9455050                 |
+------------------------+



+-------+--------+-------------------+-----------+----------+--------+-------+---------+----------+----------+-------+---------+----------------+
|user_id|video_id|event_ts           |gap_seconds|session_id|is_click|is_like|is_follow|is_comment|is_forward|is_hate|long_view|is_profile_enter|
+-------+--------+-------------------+-----------+----------+--------+-------+---------+----------+----------+-------+---------+----------------+
|12     |3209994 |2022-04-08 01:19:26|NULL       |12_1      |1       |0      |0        |0         |0         |0      |1        |0               |
|12     |1580601 |2022-04-08 04:39:32|12006      |12_2      |0       |0      |0        |0         |0         |0      |0        |0               |
|12     |3199380 |2022-04-08 04:39:32|0          |12_2      |0       |0      |0        |0         |0         |0      |0        |0               |
|12     |2351721 |2022-04-08 04:39:32|0          |12_2      |1       |0      |0        |0         |0         |0      |1     

In [95]:
session_stats = logs_sequence.groupBy("session_id", USER_COL).agg(
    F.min("event_ts").alias("session_start"),
    F.max("event_ts").alias("session_end"),
    F.count("*").alias("events"),
    F.countDistinct(ITEM_COL).alias("distinct_items"),
    *[F.sum(F.col(c).cast("long")).alias(c) for c in existing_interaction_cols],
)

session_stats.orderBy(F.desc("events")).show(20, truncate=False)

+----------+-------+-------------------+-------------------+------+--------------+--------+-------+---------+----------+----------+-------+---------+----------------+
|session_id|user_id|session_start      |session_end        |events|distinct_items|is_click|is_like|is_follow|is_comment|is_forward|is_hate|long_view|is_profile_enter|
+----------+-------+-------------------+-------------------+------+--------------+--------+-------+---------+----------+----------+-------+---------+----------------+
|879_236   |879    |2022-05-05 22:31:16|2022-05-06 02:56:48|5116  |5055          |159     |2      |1        |1         |0         |0      |120      |3               |
|879_252   |879    |2022-05-07 22:48:03|2022-05-08 03:02:53|3995  |3942          |193     |0      |0        |0         |0         |0      |141      |1               |
|879_181   |879    |2022-04-28 23:43:26|2022-04-29 03:37:57|3655  |3602          |135     |0      |0        |0         |0         |0      |110      |4               

In [96]:
daily_user_pref = logs_ts.groupBy(USER_COL, "event_date").agg(
    F.count("*").alias("interactions"),
    F.countDistinct(ITEM_COL).alias("distinct_items"),
    *[F.avg(F.col(c).cast("double")).alias(f"{c}_rate") for c in existing_interaction_cols],
)

daily_user_pref.orderBy(F.desc("interactions"), USER_COL, "event_date").show(20, truncate=False)

+-------+----------+------------+--------------+--------------------+---------------------+---------------------+---------------------+---------------------+---------------------+--------------------+---------------------+
|user_id|event_date|interactions|distinct_items|is_click_rate       |is_like_rate         |is_follow_rate       |is_comment_rate      |is_forward_rate      |is_hate_rate         |long_view_rate      |is_profile_enter_rate|
+-------+----------+------------+--------------+--------------------+---------------------+---------------------+---------------------+---------------------+---------------------+--------------------+---------------------+
|205    |2022-05-02|9023        |8728          |0.167017621633603   |0.006317189404854261 |0.0                  |0.0017732461487310208|1.108278842956888E-4 |0.004100631718940486 |0.0514241383131996  |0.0035464922974620416|
|413    |2022-04-26|8140        |7878          |0.08968058968058969 |1.2285012285012285E-4|0.0              

## Silver Layer Outputs

Silver keeps the feature groups complete and lightly cleaned. Video metadata and historical statistics are intentionally stored separately here; task-specific joins and feature selection should happen later in gold tables.


In [ ]:
SILVER_DIR = PROJECT_ROOT / "data" / "silver" / "kuairand"
silver_table_names = ["interactions", "users", "videos_basic", "videos_statistics"]

if not SILVER_DIR.exists():
    print(f"Silver directory does not exist yet: {SILVER_DIR}")
    print("Build it with: python -m recommender.data.build_silver --overwrite")
else:
    silver_tables = {}
    for name in silver_table_names:
        path = SILVER_DIR / name
        if path.exists():
            df = spark.read.parquet(str(path))
            silver_tables[name] = df
            print(f"{name}: rows={df.count():,}, columns={len(df.columns)}")
            print(df.columns[:20])
        else:
            print(f"missing: {path}")


In [ ]:
if "silver_tables" in globals():
    for name, df in silver_tables.items():
        print(f"\n{name}")
        df.printSchema()
